### Links and Resources
| Method / Command | Description |
|------------------|-------------|
| [DeltaTable.merge()](https://docs.delta.io/latest/api/python/spark/index.html) | Entry point for MERGE — returns a `DeltaMergeBuilder` to chain clauses |
| [whenMatchedUpdate()](https://docs.delta.io/latest/api/python/spark/index.html) | Update target rows that match the join condition |
| [whenMatchedUpdateAll()](https://docs.delta.io/latest/api/python/spark/index.html) | Update all columns in matched rows with source values |
| [whenMatchedDelete()](https://docs.delta.io/latest/api/python/spark/index.html) | Delete target rows that match the join condition |
| [whenNotMatchedInsert()](https://docs.delta.io/latest/api/python/spark/index.html) | Insert source rows that have no match in the target |
| [whenNotMatchedInsertAll()](https://docs.delta.io/latest/api/python/spark/index.html) | Insert all columns of unmatched source rows into target |
| [whenNotMatchedBySourceDelete()](https://docs.delta.io/latest/api/python/spark/index.html) | Delete target rows that have no matching row in the source |
| [MERGE INTO (SQL)](https://docs.databricks.com/en/sql/language-manual/delta-merge-into.html) | SQL syntax for MERGE operations |
| [operationMetrics](https://docs.databricks.com/en/delta/history.html#operation-metrics-keys) | Per-commit metrics: inserted, updated, deleted, and copied row counts |

In [ ]:
import os
if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
    from spark_setup import spark, display  # local run through Databricks Connect

# Delta Table MERGE

**MERGE** (also called *upsert*) is the most powerful mutation operation in Delta Lake.  
A single MERGE commit can **insert**, **update**, and **delete** rows atomically, driven by a join between the target table and a source dataset.

It is the standard pattern for:
- **CDC ingestion** — apply change-data-capture records from an upstream system.
- **Incremental loads** — insert new rows, update changed rows, leave unchanged rows untouched.
- **Full sync** — make the target match the source exactly, deleting anything not in the source.
- **Deduplication** — insert a row only if it does not already exist.

### Python API overview

```python
DeltaTable.merge(source_df, join_condition)   # start — returns DeltaMergeBuilder
    .whenMatchedUpdate(condition, set)         # update matching rows
    .whenMatchedUpdateAll(condition)           # update all columns from source
    .whenMatchedDelete(condition)              # delete matching rows
    .whenNotMatchedInsert(condition, values)   # insert unmatched source rows
    .whenNotMatchedInsertAll(condition)        # insert all columns of unmatched source rows
    .whenNotMatchedBySourceDelete(condition)   # delete target rows absent from source
    .execute()                                 # commit — writes one Delta log entry
```

### How clauses are evaluated

For every row in the **source**, Delta checks which target row (if any) satisfies the join condition:

```
Source row
  ├── MATCHED (join condition is true)
  │     ├── whenMatchedUpdate  ─── update the target row
  │     └── whenMatchedDelete  ─── delete the target row
  └── NOT MATCHED (no target row)
        └── whenNotMatchedInsert ── insert a new target row

Target row with no matching source row
  └── whenNotMatchedBySourceDelete ── delete the target row
```

Multiple `whenMatched` clauses are evaluated in order — the first one whose `condition` is true wins.

## Setup — Create the Target Table

The target is `product_inventory` — the same 12-row table used in earlier notebooks, recreated fresh so version numbers are predictable.

In [ ]:
from pyspark.sql import Row, functions as F
from delta.tables import DeltaTable

TABLE = "population_metrics.default.product_inventory"

target_rows = [
    Row(product_id=1,  name="Laptop Pro 15",      category="Electronics", price=1299.99, in_stock=True),
    Row(product_id=2,  name="Wireless Mouse",      category="Electronics", price=25.49,  in_stock=True),
    Row(product_id=3,  name="USB-C Hub",           category="Electronics", price=39.99,  in_stock=False),
    Row(product_id=4,  name="Standing Desk",       category="Furniture",   price=499.00, in_stock=True),
    Row(product_id=5,  name="Ergonomic Chair",     category="Furniture",   price=349.00, in_stock=True),
    Row(product_id=6,  name="Monitor Arm",         category="Furniture",   price=89.95,  in_stock=False),
    Row(product_id=7,  name="Notebook (A5)",       category="Stationery",  price=4.99,   in_stock=True),
    Row(product_id=8,  name="Ballpoint Pen 10pk",  category="Stationery",  price=2.49,   in_stock=True),
    Row(product_id=9,  name="Sticky Notes 5pk",    category="Stationery",  price=3.79,   in_stock=True),
    Row(product_id=10, name="Whiteboard Marker",   category="Stationery",  price=5.99,   in_stock=False),
    Row(product_id=11, name="Coffee Mug",          category="Kitchen",     price=12.00,  in_stock=True),
    Row(product_id=12, name="Desk Lamp LED",       category="Electronics", price=45.00,  in_stock=True),
]

spark.createDataFrame(target_rows).write.format("delta").mode("overwrite").saveAsTable(TABLE)
print(f"Target table created — version 0, 12 rows.")
display(spark.table(TABLE).orderBy("product_id"))

# 1. Simple Upsert — `whenMatchedUpdate` + `whenNotMatchedInsert`

The most common MERGE pattern: update existing rows, insert new ones.

**Source dataset:** a daily price feed containing:
- `product_id = 1` — price has changed (update).
- `product_id = 3` — also updated (`in_stock` flipped to True).
- `product_id = 13` — a brand-new product not yet in the target (insert).

Rows that exist in the target but are absent from the source (`2, 4–12`) are left untouched.

In [ ]:
source_upsert = spark.createDataFrame([
    Row(product_id=1,  name="Laptop Pro 15",   category="Electronics", price=1099.99, in_stock=True),
    Row(product_id=3,  name="USB-C Hub",        category="Electronics", price=39.99,  in_stock=True),
    Row(product_id=13, name="Mechanical Keyboard", category="Electronics", price=129.00, in_stock=True),
])

dt = DeltaTable.forName(spark, TABLE)

(
    dt.merge(
        source=source_upsert,
        condition="target.product_id = source.product_id"
    )
    .whenMatchedUpdate(set={
        "price":    "source.price",
        "in_stock": "source.in_stock",
    })
    .whenNotMatchedInsert(values={
        "product_id": "source.product_id",
        "name":       "source.name",
        "category":   "source.category",
        "price":      "source.price",
        "in_stock":   "source.in_stock",
    })
    .execute()
)

print("After upsert MERGE (version 1):")
display(spark.table(TABLE).orderBy("product_id"))
# product_id 1  → price lowered to 1099.99
# product_id 3  → in_stock flipped to true
# product_id 13 → new row inserted

# 2. Conditional Update — Only Change Rows That Actually Differ

By default, `whenMatchedUpdate` rewrites the target row even if the source values are identical.  
Adding a `condition` to the `whenMatched` clause limits rewrites to rows where at least one value changed.

This is important for performance: fewer updated rows → fewer Parquet files rewritten → smaller `numCopiedRows` in `operationMetrics`.

**Scenario:** Ingest a price feed where only some rows are actually different from what is in the target.

In [ ]:
# Source has 4 rows: 2 unchanged, 2 with real changes
source_conditional = spark.createDataFrame([
    Row(product_id=4,  name="Standing Desk",   category="Furniture",   price=499.00, in_stock=True),   # SAME — no update needed
    Row(product_id=5,  name="Ergonomic Chair", category="Furniture",   price=399.00, in_stock=True),   # price changed
    Row(product_id=7,  name="Notebook (A5)",   category="Stationery",  price=4.99,   in_stock=True),   # SAME — no update needed
    Row(product_id=8,  name="Ballpoint Pen 10pk", category="Stationery", price=2.99, in_stock=True),   # price changed
])

dt = DeltaTable.forName(spark, TABLE)

(
    dt.merge(
        source=source_conditional,
        condition="target.product_id = source.product_id"
    )
    .whenMatchedUpdate(
        # Only rewrite rows where price or in_stock actually changed
        condition="target.price <> source.price OR target.in_stock <> source.in_stock",
        set={
            "price":    "source.price",
            "in_stock": "source.in_stock",
        }
    )
    .execute()
)

print("After conditional MERGE (version 2) — only rows 5 and 8 changed:")
display(
    spark.table(TABLE)
    .filter(F.col("product_id").isin(4, 5, 7, 8))
    .orderBy("product_id")
)

# 3. `whenMatchedUpdateAll` and `whenNotMatchedInsertAll` — Shorthand

When the source and target share the **same schema**, you can use the `All` shorthands instead of listing every column in `set` or `values`.

| Verbose | Shorthand |
|---------|-----------|
| `whenMatchedUpdate(set={"col1": "source.col1", ...})` | `whenMatchedUpdateAll()` |
| `whenNotMatchedInsert(values={"col1": "source.col1", ...})` | `whenNotMatchedInsertAll()` |

**Scenario:** A new source batch with a mix of updates and new products — all columns match the target schema.

In [ ]:
source_all = spark.createDataFrame([
    Row(product_id=6,  name="Monitor Arm",      category="Furniture",   price=99.95,  in_stock=True),   # update — restocked
    Row(product_id=10, name="Whiteboard Marker", category="Stationery", price=6.49,   in_stock=True),   # update — restocked + new price
    Row(product_id=14, name="Webcam HD",         category="Electronics", price=79.00, in_stock=True),   # insert
])

dt = DeltaTable.forName(spark, TABLE)

(
    dt.merge(
        source=source_all,
        condition="target.product_id = source.product_id"
    )
    .whenMatchedUpdateAll()         # copy every source column to the matching target row
    .whenNotMatchedInsertAll()      # insert the full source row when no match exists
    .execute()
)

print("After UpdateAll / InsertAll MERGE (version 3):")
display(
    spark.table(TABLE)
    .filter(F.col("product_id").isin(6, 10, 14))
    .orderBy("product_id")
)

# 4. `whenMatchedDelete` — Delete Matching Rows

`whenMatchedDelete` removes target rows whose `product_id` appears in the source.  
This is the MERGE equivalent of `DELETE FROM target WHERE product_id IN (SELECT product_id FROM source)`.

**Scenario:** A "discontinue" feed lists products that should be removed from inventory.

In [ ]:
# Source is a list of product IDs to discontinue
source_discontinue = spark.createDataFrame([
    Row(product_id=11),   # Coffee Mug — discontinued
    Row(product_id=12),   # Desk Lamp LED — discontinued
])

dt = DeltaTable.forName(spark, TABLE)

(
    dt.merge(
        source=source_discontinue,
        condition="target.product_id = source.product_id"
    )
    .whenMatchedDelete()
    .execute()
)

print("After delete-on-match MERGE (version 4):")
display(spark.table(TABLE).orderBy("product_id"))
# product_id 11 and 12 should be gone

# 5. `whenNotMatchedBySourceDelete` — Full Table Sync

`whenNotMatchedBySourceDelete` deletes target rows that have **no corresponding row in the source**.  
Combined with `whenMatchedUpdateAll` and `whenNotMatchedInsertAll`, this produces a **full sync** — after the MERGE the target contains exactly the same rows as the source.

This is the standard pattern for:
- Replacing a dimension table with a new snapshot.
- Synchronising a target table with a system-of-record feed.

**Scenario:** The authoritative product catalogue is published as a full snapshot — apply it to the target.

In [ ]:
# The authoritative full snapshot — only 5 products should exist after sync
source_snapshot = spark.createDataFrame([
    Row(product_id=1,  name="Laptop Pro 15",      category="Electronics", price=1099.99, in_stock=True),
    Row(product_id=2,  name="Wireless Mouse",      category="Electronics", price=25.49,  in_stock=True),
    Row(product_id=4,  name="Standing Desk",       category="Furniture",   price=499.00, in_stock=True),
    Row(product_id=5,  name="Ergonomic Chair",     category="Furniture",   price=399.00, in_stock=True),
    Row(product_id=20, name="Smart Speaker",       category="Electronics", price=89.00,  in_stock=True),  # brand new
])

dt = DeltaTable.forName(spark, TABLE)

(
    dt.merge(
        source=source_snapshot,
        condition="target.product_id = source.product_id"
    )
    .whenMatchedUpdateAll()             # update existing rows if source differs
    .whenNotMatchedInsertAll()          # insert rows present in source but not in target
    .whenNotMatchedBySourceDelete()     # delete rows in target but absent from source
    .execute()
)

print("After full-sync MERGE (version 5) — target should mirror source exactly:")
display(spark.table(TABLE).orderBy("product_id"))
# Only product_ids 1, 2, 4, 5, 20 should remain

# 6. SQL MERGE INTO Syntax

SQL `MERGE INTO` is fully equivalent to the Python API.  
It is useful in `%sql` cells, dbt models, or any context where you prefer declarative SQL over Python.

```sql
MERGE INTO target
USING source
ON target.key = source.key
WHEN MATCHED [AND extra_condition] THEN UPDATE SET ...
WHEN MATCHED [AND extra_condition] THEN DELETE
WHEN NOT MATCHED [BY TARGET] [AND extra_condition] THEN INSERT ...
WHEN NOT MATCHED BY SOURCE [AND extra_condition] THEN DELETE
```

**Scenario:** Apply a correction feed using pure SQL — update product names and prices for two rows.

In [ ]:
# Register the source DataFrame as a temporary view so SQL can reference it
source_corrections = spark.createDataFrame([
    Row(product_id=1, name="Laptop Pro 15 (2025)", category="Electronics", price=1049.99, in_stock=True),
    Row(product_id=2, name="Wireless Mouse",        category="Electronics", price=22.99,   in_stock=True),
    Row(product_id=21, name="Desk Organiser",       category="Accessories", price=19.99,   in_stock=True),
])
source_corrections.createOrReplaceTempView("source_corrections")

spark.sql(f"""
    MERGE INTO {TABLE} AS target
    USING source_corrections AS source
    ON target.product_id = source.product_id
    WHEN MATCHED THEN
        UPDATE SET
            target.name     = source.name,
            target.price    = source.price,
            target.in_stock = source.in_stock
    WHEN NOT MATCHED THEN
        INSERT (product_id, name, category, price, in_stock)
        VALUES (source.product_id, source.name, source.category, source.price, source.in_stock)
""")

print("After SQL MERGE INTO (version 6):")
display(spark.table(TABLE).orderBy("product_id"))

# 7. Deduplication — Insert Only If Not Already Present

MERGE is the standard way to implement **exactly-once inserts**: insert a row only when no matching key exists in the target.  
This guards against duplicates from retried batch jobs or streaming micro-batches.

The pattern uses only `whenNotMatchedInsertAll()` — no `whenMatched` clause means existing rows are never touched.

In [ ]:
# Simulated retry: source contains a mix of already-ingested and genuinely new rows
source_retry = spark.createDataFrame([
    Row(product_id=1,  name="Laptop Pro 15 (2025)", category="Electronics", price=1049.99, in_stock=True),  # already exists
    Row(product_id=2,  name="Wireless Mouse",        category="Electronics", price=22.99,   in_stock=True),  # already exists
    Row(product_id=22, name="Cable Organiser",       category="Accessories", price=9.99,    in_stock=True),  # genuinely new
])

dt = DeltaTable.forName(spark, TABLE)

(
    dt.merge(
        source=source_retry,
        condition="target.product_id = source.product_id"
    )
    # No whenMatchedUpdate — existing rows are intentionally left unchanged
    .whenNotMatchedInsertAll()
    .execute()
)

print("After dedup MERGE (version 7) — only product_id 22 should be new:")
display(spark.table(TABLE).orderBy("product_id"))

# 8. MERGE `operationMetrics` Deep Dive

Like `UPDATE`, every MERGE commit records detailed row-level and file-level counters in `operationMetrics`.  
These metrics are the primary tool for auditing what a MERGE actually did.

| Metric | Meaning |
|--------|---------|
| `numSourceRows` | Rows in the source DataFrame evaluated by the MERGE |
| `numTargetRowsInserted` | Target rows inserted by `whenNotMatchedInsert` |
| `numTargetRowsUpdated` | Target rows updated by `whenMatchedUpdate` |
| `numTargetRowsDeleted` | Target rows deleted by `whenMatchedDelete` or `whenNotMatchedBySourceDelete` |
| `numTargetRowsCopied` | Rows in the same Parquet files as changed rows that were rewritten unchanged — the hidden I/O cost |
| `numOutputRows` | Total rows written to new Parquet files (`inserted + updated + copied`) |
| `numTargetFilesAdded` | New Parquet files written |
| `numTargetFilesRemoved` | Old Parquet files replaced (still on storage until VACUUM) |

In [ ]:
dt = DeltaTable.forName(spark, TABLE)

# Pull every MERGE version with its full metric set
metrics_df = (
    dt.history()
    .filter("operation = 'MERGE'")
    .select(
        "version",
        "timestamp",
        F.col("operationParameters")["predicate"].alias("joinCondition"),
        F.col("operationMetrics")["numSourceRows"].alias("sourceRows"),
        F.col("operationMetrics")["numTargetRowsInserted"].alias("inserted"),
        F.col("operationMetrics")["numTargetRowsUpdated"].alias("updated"),
        F.col("operationMetrics")["numTargetRowsDeleted"].alias("deleted"),
        F.col("operationMetrics")["numTargetRowsCopied"].alias("copied"),
        F.col("operationMetrics")["numTargetFilesAdded"].alias("filesAdded"),
        F.col("operationMetrics")["numTargetFilesRemoved"].alias("filesRemoved"),
    )
    .orderBy("version")
)

display(metrics_df)

# 9. Full History After All MERGEs

Each MERGE is a single commit in the Delta log regardless of how many rows it touches.  
The operation type is always `MERGE` — subclauses (update/insert/delete) are recorded inside `operationMetrics`, not as separate versions.

In [ ]:
dt = DeltaTable.forName(spark, TABLE)

display(
    dt.history()
    .select("version", "timestamp", "operation", "operationParameters")
    .orderBy("version")
)

# 10. Time Travel After MERGE

MERGE rewrites the affected Parquet files (same mechanism as UPDATE — no Deletion Vectors).  
Pre-merge versions remain readable until `VACUUM` removes their files.

This is valuable for:
- **Auditing** — compare the table before and after a MERGE to see exactly what changed.
- **Recovery** — restore a pre-merge snapshot if the source data was wrong.

In [ ]:
# Version 0 — original 12-row table before any MERGE
df_v0 = spark.sql(f"SELECT * FROM {TABLE} VERSION AS OF 0 ORDER BY product_id")
print(f"Version 0 row count: {df_v0.count()}")
display(df_v0)

In [ ]:
# Version 5 — after full-sync MERGE (only 5 rows should remain)
df_v5 = spark.sql(f"SELECT * FROM {TABLE} VERSION AS OF 5 ORDER BY product_id")
print(f"Version 5 row count (after full-sync): {df_v5.count()}")
display(df_v5)

In [ ]:
# Compare current state to version 0 to see everything that changed
current = spark.table(TABLE).select("product_id", "name", "price").alias("current")
original = spark.sql(f"SELECT product_id, name, price FROM {TABLE} VERSION AS OF 0").alias("original")

# Rows that exist in current but not in v0 (net-new rows)
net_new = current.join(original, on="product_id", how="left_anti")
print("Rows in current but not in version 0 (net new):")
display(net_new.orderBy("product_id"))

# Rows in v0 but gone from current (deleted over time)
removed = original.join(current, on="product_id", how="left_anti")
print("Rows in version 0 but absent from current (removed):")
display(removed.orderBy("product_id"))

# 11. OPTIMIZE After Repeated MERGEs

Each MERGE writes new Parquet files for every row it touches.  
Repeated MERGEs on overlapping keys leave behind many small files — each containing only the rows that were affected by that particular batch.

### What fragmentation looks like

```
After MERGE batch 1:  file_A (rows 1,2,3), file_B (rows 4,5)
After MERGE batch 2:  file_A (rows 1,2,3), file_C (rows 4_updated, 5, 6_new)
After MERGE batch 3:  file_A (rows 1,2,3), file_C (rows 4,5,6), file_D (rows 7_new)
```

Over time there are many small files (`file_C`, `file_D`, ...) and one large original file.  
OPTIMIZE collapses them back into a small number of right-sized files.

### When to run OPTIMIZE

| Pattern | Recommendation |
|---------|----------------|
| Hourly micro-batch MERGEs | Run OPTIMIZE daily or after N batches |
| Daily full-snapshot MERGE | Run OPTIMIZE after each MERGE |
| Streaming MERGE | Auto-optimise is enabled by default on Databricks |
| Large one-off MERGE (millions of rows) | Run OPTIMIZE + ZORDER immediately after |

In [ ]:
# File count and size before OPTIMIZE
dt = DeltaTable.forName(spark, TABLE)
before = dt.detail().select("numFiles", "sizeInBytes").collect()[0]
print("Before OPTIMIZE:")
print(f"  numFiles   : {before['numFiles']}")
print(f"  sizeInBytes: {before['sizeInBytes']}")

In [ ]:
# Compact files — also resolves any pending deletion vectors
display(spark.sql(f"OPTIMIZE {TABLE}"))

dt = DeltaTable.forName(spark, TABLE)
after = dt.detail().select("numFiles", "sizeInBytes").collect()[0]
print("After OPTIMIZE:")
print(f"  numFiles   : {after['numFiles']}")
print(f"  sizeInBytes: {after['sizeInBytes']}")

# Summary

## MERGE Clause Reference

| Clause | Python API | SQL | When fires |
|--------|-----------|-----|------------|
| Update matched rows | `whenMatchedUpdate(condition, set)` | `WHEN MATCHED THEN UPDATE SET ...` | Source key exists in target |
| Update all columns | `whenMatchedUpdateAll(condition)` | `WHEN MATCHED THEN UPDATE SET *` | Same — shorthand |
| Delete matched rows | `whenMatchedDelete(condition)` | `WHEN MATCHED THEN DELETE` | Source key exists in target |
| Insert unmatched rows | `whenNotMatchedInsert(condition, values)` | `WHEN NOT MATCHED THEN INSERT ...` | Source key absent from target |
| Insert all columns | `whenNotMatchedInsertAll(condition)` | `WHEN NOT MATCHED THEN INSERT *` | Same — shorthand |
| Delete unmatched target | `whenNotMatchedBySourceDelete(condition)` | `WHEN NOT MATCHED BY SOURCE THEN DELETE` | Target key absent from source |

## Common Patterns

| Pattern | Clauses used |
|---------|-------------|
| **Upsert** | `whenMatchedUpdate` + `whenNotMatchedInsert` |
| **Conditional upsert** | `whenMatchedUpdate(condition=...)` + `whenNotMatchedInsert` |
| **Full sync** | `whenMatchedUpdateAll` + `whenNotMatchedInsertAll` + `whenNotMatchedBySourceDelete` |
| **Delete-on-match** | `whenMatchedDelete` |
| **Deduplication** | `whenNotMatchedInsertAll` only |

## MERGE vs UPDATE vs DELETE — Internal Mechanics

| Operation | Uses Deletion Vectors? | Rewrites Parquet files? | Orphan files created? |
|-----------|----------------------|------------------------|----------------------|
| DELETE | Yes (if enabled) | No | No |
| UPDATE | No | Yes — affected files | Yes |
| MERGE | No | Yes — affected files | Yes |

MERGE and UPDATE share the same file-rewrite model. The `numTargetRowsCopied` metric reflects the cost of co-located rows being rewritten even though their values did not change.

## Lifecycle

```
MERGE
  └─► Affected Parquet files rewritten — old files become orphans
       └─► history() records version, join condition, and full row/file metrics
            └─► OPTIMIZE (after N batches or daily)
                  └─► Compacts small files from incremental MERGEs
                       └─► VACUUM (after retention period)
                             └─► Orphan files permanently removed
```